# Interview transcripts, coded segments, opportunity centre records, and interview instrument items from City of Johannesburg Opportunity Centres fieldwork, October-December 2023 Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, inspecting, and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.d0d2-39r6/fair2.json`

This notebook follows a reusable pattern:
- load Croissant metadata,
- inspect available record sets and fields using their `@id` values,
- extract records into pandas DataFrames,
- perform lightweight exploratory processing,
- visualize distributions and relationships when suitable.

> Important: all references to record sets and fields in this notebook are made using their Croissant `@id` values whenever those entities are available in the metadata.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant JSON-LD URL and inspect the high-level dataset description.

In this section, we keep `dataset.metadata` as a single object and access its attributes safely.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.d0d2-39r6/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset name:')
print(getattr(metadata, 'name', 'N/A'))
print('\nDescription:')
print(getattr(metadata, 'description', 'N/A'))
print('\nIdentifier:')
print(getattr(metadata, 'identifier', 'N/A'))
print('\nLicense:')
print(getattr(metadata, 'license', 'N/A'))
print('\nVersion:')
print(getattr(metadata, 'version', 'N/A'))

In [ ]:
# Optional: inspect the metadata object structure
print(type(metadata))
print('\nAvailable attributes on metadata:')
print([attr for attr in dir(metadata) if not attr.startswith('_')][:100])

## 2. Data Overview
Review the available record sets and fields exposed by the Croissant metadata.

Because Croissant datasets may differ in how metadata is represented, the helper functions below try common access patterns while keeping references centered on `@id` values.

In [ ]:
def to_serializable_metadata(obj):
    """Convert metadata to a Python structure if supported by the object."""
    if hasattr(obj, 'to_json'):
        converted = obj.to_json()
        if isinstance(converted, str):
            try:
                return json.loads(converted)
            except Exception:
                return converted
        return converted
    return obj

metadata_json = to_serializable_metadata(metadata)
type(metadata_json)

In [ ]:
def get_recordsets_from_metadata(metadata_obj, metadata_json_obj):
    """Best-effort extraction of record sets from metadata."""
    if hasattr(metadata_obj, 'record_sets'):
        rs = getattr(metadata_obj, 'record_sets')
        if rs is not None:
            return rs
    if hasattr(metadata_obj, 'recordSet'):
        rs = getattr(metadata_obj, 'recordSet')
        if rs is not None:
            return rs
    if isinstance(metadata_json_obj, dict):
        if 'recordSet' in metadata_json_obj:
            return metadata_json_obj['recordSet']
        if 'record_sets' in metadata_json_obj:
            return metadata_json_obj['record_sets']
    return []

record_sets_meta = get_recordsets_from_metadata(metadata, metadata_json)
print(f'Number of discovered record sets: {len(record_sets_meta) if hasattr(record_sets_meta, "__len__") else 0}')

In [ ]:
def get_attr_or_key(obj, name, default=None):
    if hasattr(obj, name):
        return getattr(obj, name)
    if isinstance(obj, dict):
        return obj.get(name, default)
    return default

record_set_rows = []

for rs in record_sets_meta:
    rs_id = get_attr_or_key(rs, 'id', None) or get_attr_or_key(rs, '@id', None)
    rs_name = get_attr_or_key(rs, 'name', None)
    rs_desc = get_attr_or_key(rs, 'description', None)
    fields = get_attr_or_key(rs, 'fields', None) or get_attr_or_key(rs, 'field', None) or []
    record_set_rows.append({
        'record_set_id': rs_id,
        'record_set_name': rs_name,
        'record_set_description': rs_desc,
        'field_count': len(fields) if hasattr(fields, '__len__') else 0
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

If the dataset defines record sets, the next cell expands their fields and displays their field `@id`s, names, and data types.

In [ ]:
field_rows = []

for rs in record_sets_meta:
    rs_id = get_attr_or_key(rs, 'id', None) or get_attr_or_key(rs, '@id', None)
    rs_name = get_attr_or_key(rs, 'name', None)
    fields = get_attr_or_key(rs, 'fields', None) or get_attr_or_key(rs, 'field', None) or []
    for field in fields:
        field_id = get_attr_or_key(field, 'id', None) or get_attr_or_key(field, '@id', None)
        field_name = get_attr_or_key(field, 'name', None)
        field_desc = get_attr_or_key(field, 'description', None)
        field_dtype = get_attr_or_key(field, 'data_types', None) or get_attr_or_key(field, 'dataType', None)
        source = get_attr_or_key(field, 'source', None)
        field_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_id': field_id,
            'field_name': field_name,
            'field_description': field_desc,
            'field_data_type': str(field_dtype),
            'field_source': str(source)
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(50)

In [ ]:
# Show full field inventory, if available
if not fields_df.empty:
    display(fields_df.sort_values(['record_set_id', 'field_id']).reset_index(drop=True))
else:
    print('No record sets or fields were exposed in the loaded metadata. This dataset may currently publish descriptive metadata without tabular record sets.')

The following example shows how records would be iterated from a specific record set using its `@id`.

This cell is defensive: it only runs when at least one record set `@id` is available.

In [ ]:
record_set_ids = record_sets_df['record_set_id'].dropna().tolist() if not record_sets_df.empty else []

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f'Previewing records from record set @id: {example_record_set_id}')
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print('No record set @id values are available to preview records.')

## 3. Data Extraction
Load each discovered record set into a pandas DataFrame using its record set `@id`.

This mirrors the template, while keeping the notebook flexible for datasets that may have zero, one, or many record sets.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} rows from record set: {record_set_id}')
    except Exception as e:
        print(f'Could not load record set {record_set_id}: {e}')

In [ ]:
if dataframes:
    sample_record_set_id = next(iter(dataframes.keys()))
    print(f'Sample record set @id: {sample_record_set_id}')
    print('Columns:')
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print('No DataFrames were created. This is consistent with metadata that does not currently expose any recordSet entries.')

It is also helpful to summarize dataset shapes across all loaded record sets.

In [ ]:
df_summary_rows = []

for rs_id, df in dataframes.items():
    df_summary_rows.append({
        'record_set_id': rs_id,
        'rows': df.shape[0],
        'columns': df.shape[1]
    })

df_summary = pd.DataFrame(df_summary_rows)
df_summary if not df_summary.empty else print('No loaded record sets to summarize.')

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps to the extracted data.

Since Croissant datasets can vary widely, this section dynamically:
- finds numeric fields,
- filters on a numeric threshold,
- normalizes a numeric field,
- groups by a categorical field when available.

If the dataset does not currently expose record sets, the code will gracefully report that instead of failing.

In [ ]:
def choose_analysis_dataframe(dataframes_dict):
    if not dataframes_dict:
        return None, None
    # Prefer the widest non-empty DataFrame
    ranked = sorted(
        dataframes_dict.items(),
        key=lambda kv: (kv[1].shape[0] > 0, kv[1].shape[1], kv[1].shape[0]),
        reverse=True
    )
    return ranked[0]

analysis_record_set_id, analysis_df = choose_analysis_dataframe(dataframes)

if analysis_df is not None:
    print(f'Using record set @id for EDA: {analysis_record_set_id}')
    print(analysis_df.head())
else:
    print('No DataFrame is available for EDA.')

In [ ]:
if analysis_df is not None and not analysis_df.empty:
    numeric_columns = analysis_df.select_dtypes(include='number').columns.tolist()
    object_columns = analysis_df.select_dtypes(include='object').columns.tolist()
    print('Numeric columns:', numeric_columns)
    print('Text/categorical candidate columns:', object_columns[:20])
else:
    numeric_columns = []
    object_columns = []
    print('No populated DataFrame available to inspect column types.')

In [ ]:
if analysis_df is not None and not analysis_df.empty and numeric_columns:
    numeric_field_id = numeric_columns[0]
    threshold = analysis_df[numeric_field_id].median()
    filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()

    print(f'Numeric field selected: {numeric_field_id}')
    print(f'Filtering rows where {numeric_field_id} > {threshold}')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field_id].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f'{numeric_field_id}_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / std_value
    else:
        filtered_df[f'{numeric_field_id}_normalized'] = 0.0

    print('Normalized numeric field preview:')
    display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    group_field_id = None
    for candidate in object_columns:
        if analysis_df[candidate].nunique(dropna=True) > 1 and analysis_df[candidate].nunique(dropna=True) < max(20, len(analysis_df) * 0.5):
            group_field_id = candidate
            break

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f'Grouped mean of {numeric_field_id} by {group_field_id}:')
        display(grouped_df.head())
    else:
        print('No suitable grouping field was found for grouped analysis.')
else:
    print('No numeric fields were available, so numeric filtering and normalization were skipped.')

For text-heavy datasets, simple text-derived features can also be useful. The next cell computes a character-length summary for one text-like column, if available.

In [ ]:
if analysis_df is not None and not analysis_df.empty and object_columns:
    text_field_id = None
    for candidate in object_columns:
        non_null = analysis_df[candidate].dropna()
        if len(non_null) > 0:
            avg_len = non_null.astype(str).str.len().mean()
            if avg_len > 20:
                text_field_id = candidate
                break

    if text_field_id is not None:
        text_df = analysis_df[[text_field_id]].dropna().copy()
        text_df[f'{text_field_id}_char_length'] = text_df[text_field_id].astype(str).str.len()
        print(f'Text-derived feature created from field: {text_field_id}')
        display(text_df.head())
        print(text_df[f'{text_field_id}_char_length'].describe())
    else:
        print('No sufficiently long text field was detected for text-length analysis.')
else:
    print('No object/text columns were available for text feature engineering.')

## 5. Visualization
Visualize numeric and categorical patterns when they are available.

The plotting code is dynamic so it can adapt to whichever fields were loaded from the Croissant record sets.

In [ ]:
if analysis_df is not None and not analysis_df.empty and numeric_columns:
    numeric_field_id = numeric_columns[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(analysis_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Numeric histogram skipped because no numeric column was available.')

In [ ]:
if analysis_df is not None and not analysis_df.empty and object_columns:
    category_field_id = None
    for candidate in object_columns:
        nunique = analysis_df[candidate].nunique(dropna=True)
        if 1 < nunique <= 15:
            category_field_id = candidate
            break

    if category_field_id is not None:
        plt.figure(figsize=(10, 4))
        value_counts = analysis_df[category_field_id].astype(str).value_counts().head(10)
        sns.barplot(x=value_counts.index, y=value_counts.values)
        plt.title(f'Top categories in {category_field_id}')
        plt.xlabel(category_field_id)
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('Categorical bar plot skipped because no suitable low-cardinality text field was available.')
else:
    print('Categorical visualization skipped because no text/categorical columns were available.')

In [ ]:
if analysis_df is not None and not analysis_df.empty and len(numeric_columns) >= 2:
    x_field_id, y_field_id = numeric_columns[:2]
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=analysis_df, x=x_field_id, y=y_field_id)
    plt.title(f'{y_field_id} vs {x_field_id}')
    plt.tight_layout()
    plt.show()
else:
    print('Scatter plot skipped because fewer than two numeric columns were available.')

## 6. Conclusion
This notebook demonstrated how to:
- load a dataset from a Croissant schema URL with `mlcroissant`,
- inspect metadata and discover record sets and fields by `@id`,
- extract record sets into pandas DataFrames,
- apply dynamic exploratory processing for numeric and text fields,
- generate basic visualizations when suitable columns are present.

### Notes for this dataset
- The descriptive metadata for this FAIR^2 package clearly documents the study context, collection methods, limitations, and intended use cases.
- In the supplied metadata snapshot, `recordSet` appears empty. If the published Croissant endpoint is later updated with record sets, the extraction and EDA cells in this notebook should work without major structural changes.
- Because this dataset contains qualitative and context-sensitive materials, downstream analysis should account for privacy, sampling limits, transcription caveats, and the municipality-specific context.

You can extend this notebook by:
- selecting a specific record set `@id` once available,
- using field `@id`s to build targeted analyses,
- joining related record sets on shared identifiers,
- adding domain-specific qualitative text analysis.